In [ ]:
import pandas as pd
import pyfixest as pf

df = pd.read_parquet("https://raw.githubusercontent.com/anzonyquispe/did_book/main/cc_xd_didtextbook_2025_9_30/Data%20sets/Moser%20and%20Voena%202012/moser_voena_didtextbook.parquet")



OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


                twea |    0.2882615   0.0388871      7.41     0.0000
     C(year)[T.1901] |    0.0059327   0.0077529      0.77     0.4442
     C(year)[T.1902] |    0.0059327   0.0083762      0.71     0.4788
     C(year)[T.1903] |    0.0347682   0.0087677      3.97     0.0001
     C(year)[T.1904] |    0.0184879   0.0086477      2.14     0.0326
     C(year)[T.1905] |    0.0211093   0.0090630      2.33     0.0199
     C(year)[T.1906] |    0.0430464   0.0092489      4.65     0.0000
     C(year)[T.1907] |    0.1019592   0.0104486      9.76     0.0000
     C(year)[T.1908] |    0.0583609   0.0099680      5.85     0.0000
     C(year)[T.1909] |    0.0867826   0.0108691      7.98     0.0000
     C(year)[T.1910] |    0.0584989   0.0098129      5.96     0.0000
     C(year)[T.1911] |    0.0866446   0.0109834      7.89     0.0000
     C(year)[T.1912] |    0.1092715   0.0108980     10.03     0.0000
     C(year)[T.1913] |    0.0685706   0.0112787      6.08     0.0000
     C(year)[T.1914] |    0.153145

In [8]:
# Example 1
m1 = pf.feols('patents ~ twea + i(year) | subclass', data=df, vcov={'CRV1': 'subclass'})
m1.summary()


###

Estimation:  OLS
Dep. var.: patents, Fixed effects: subclass
Inference:  CRV1
Observations:  289920

| Coefficient     |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:----------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| twea            |      0.288 |        0.039 |     7.413 |      0.000 |  0.212 |   0.364 |
| C(year)[T.1901] |      0.006 |        0.008 |     0.765 |      0.444 | -0.009 |   0.021 |
| C(year)[T.1902] |      0.006 |        0.008 |     0.708 |      0.479 | -0.010 |   0.022 |
| C(year)[T.1903] |      0.035 |        0.009 |     3.966 |      0.000 |  0.018 |   0.052 |
| C(year)[T.1904] |      0.018 |        0.009 |     2.138 |      0.033 |  0.002 |   0.035 |
| C(year)[T.1905] |      0.021 |        0.009 |     2.329 |      0.020 |  0.003 |   0.039 |
| C(year)[T.1906] |      0.043 |        0.009 |     4.654 |      0.000 |  0.025 |   0.061 |
| C(year)[T.1907] |      0.102 |        0.010 |     9.758 |      0

In [9]:
# for name in c1.index:
#     print(f"{name:>20s} | {c1[name]:>12.7f}  {s1[name]:>10.7f}  {t1[name]:>8.2f}   {p1[name]:>8.4f}")

# Example 2
m2 = pf.feols('patents ~ twea | subclass + year', data=df, vcov={'CRV1': 'subclass'})
m2.summary()

###

Estimation:  OLS
Dep. var.: patents, Fixed effects: subclass+year
Inference:  CRV1
Observations:  289920

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| twea          |      0.288 |        0.039 |     7.413 |      0.000 |  0.212 |   0.364 |
---
RMSE: 0.978 R2: 0.479 R2 Within: 0.001 


In [ ]:
# pip install did-had
import pandas as pd
from did_had import DidHad
import pandas as pd

df = pd.read_parquet("https://raw.githubusercontent.com/Credible-Answers/did_book/main/cc_xd_didtextbook_2025_9_30/Data%20sets/Pierce%20and%20Schott%202016/pierce_schott_didtextbook.parquet")

# --- Construct lemp for post-treatment years ---------------------------------
df["lemp2001"] = df["lemp2000"] + df["delta2001"]
df["lemp2002"] = df["lemp2000"] + df["delta2002"]
df["lemp2004"] = df["lemp2000"] + df["delta2004"]
df["lemp2005"] = df["lemp2000"] + df["delta2005"]

# --- Reshape to panel (industry × year) --------------------------------------
lemp_cols = [c for c in df.columns if c.startswith("lemp") and c[4:].isdigit()]

df_long = (
    df[["indusid", "ntrgap"] + lemp_cols]
    .melt(
        id_vars=["indusid", "ntrgap"],
        value_vars=lemp_cols,
        var_name="year",
        value_name="lemp",
    )
    .assign(
        year=lambda d: d["year"].str.replace("lemp", "", regex=False).astype(int),
        ntrgap=lambda d: d["ntrgap"].where(d["year"] > 2000, 0),
    )
    .reset_index(drop=True)
)
model = DidHad(kernel='tri')
results = model.fit(
    df=df_long, outcome='lemp', group='indusid',
    time='year', treatment='ntrgap', effects=4, placebo=3)
print(results)

DID-HAD Estimation Results
Number of groups: 103
Number of periods: 8
Adoption period (F): 2001
Kernel: tri
Bandwidth selection: mse-dpi
Confidence level: 95%
Dynamic effects: False

---------------------------------------------------------------------------
                          Effect Estimates                      QUG* Test
         --------------------------------------------------- ---------------
          Estimate       SE     LB.CI     UB.CI     N      BW    N.BW        T    p.val
Effect_1   0.00727  0.07875  -0.23870   0.07000   103 0.14654     21  6.14989  0.13986
Effect_2  -0.13361  0.12500  -0.58035  -0.09034   103 0.14654     21  6.14989  0.13986
Effect_3   0.03507  0.59346  -1.60353   0.72278   103 0.14654     21  6.14989  0.13986
Effect_4  -0.02173  0.61406  -1.89858   0.50851   103 0.14654     21  6.14989  0.13986
*Quasi-Untreated Group

--------------------------------------------------------------
                           Placebo Estimates
          ------------

In [1]:
import pandas as pd
import polars as pl
from did_multiplegt_dyn import DidMultiplegtDyn
df = pd.read_parquet("https://raw.githubusercontent.com/Credible-Answers/did_book/main/cc_xd_didtextbook_2025_9_30/Data%20sets/Gentzkow%20et%20al%202011/gentzkowetal_didtextbook.parquet")
df_pl = pl.from_pandas(df)
pd.set_option('display.float_format', lambda x: f'{x:.7f}')
for ell in [1, 2, 4]:
    model = DidMultiplegtDyn(df=df_pl, outcome="prestout", group="cnty90",
        time="year", treatment="numdailies", effects=ell, cluster="cnty90",
        design=[0.8, "console"])
    model.fit(); model.summary()


  Detection of treatment paths - 1 periods after first switch
            #Groups    %Groups       l=0       l=1
TreatPath1      721 64.0319716 0.0000000 1.0000000
TreatPath2      139 12.3445826 0.0000000 2.0000000
TreatPath3       54  4.7957371 2.0000000 3.0000000
TreatPath4       54  4.7957371 1.0000000 2.0000000
Treatment paths detected in switching groups: 1126
Total % shown: 85.97%

Design interpretation (first row):
  721 groups started with treatment 0.0 and then experienced treatment [np.float64(1.0)]
             Estimation of treatment effects: Event-study effects
               Block   Estimate        SE      LB CI     UB CI            N    Switchers          N.w  Switchers.w
            Effect_1  0.0144244 0.0042477  0.0060992 0.0227497 5674.0000000 1119.0000000 5674.0000000 1119.0000000
Average_Total_Effect  0.0120186 0.0035392  0.0050819 0.0189552 5674.0000000 1119.0000000 5674.0000000 1119.0000000
           Placebo_1 -0.0005025 0.0051322 -0.0105615 0.0095565 4471.00000